#autentications

In [ ]:
hugging_face_token1 = "your_hugging_face_token"
Bot_Token = "your_telegram_token"
Chat_Id = "Telegram_Chat_Id"
aiml_token = "your_aiml_token"
openrouter_token = "your_openrouter_token"
opeiai_token = "your_openai_token"


In [ ]:
# chose between datasets
# FLAG='humaneval'
FLAG = "mbpp"

# for sending results to telegram

FLAG_TELE = "True"
# FLAG_TELE='False'


In [ ]:
# FLAG_Model='opeiai'
FLAG_Model = "hugging_face"


#dependencites

In [ ]:
!pip install python-telegram-bot --upgrade
!pip install tiktoken
!pip install openai
!pip install langchain-text-splitters
!pip install langgraph
!pip install sacrebleu
!pip install -U sentence-transformers


#imports

In [ ]:
import torch

torch.cuda.is_available()
from accelerate import Accelerator
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

# import shap
import matplotlib.pyplot as plt
from IPython.core.display import HTML
import pandas as pd

torch.cuda.empty_cache()
accelerator = Accelerator()
accelerator.free_memory()

from transformers.utils import logging

logging.set_verbosity_error()
import logging

logging.getLogger("shap").setLevel(logging.ERROR)
# Cast model weights to float32


import torch
import torch.nn.functional as F


import os
import json

import torch
import pandas as pd
import json


#dataset

In [ ]:
# heman eval

if FLAG == "humaneval":
    # Load the HumanEval dataset using parquetA
    dataset = pd.read_parquet(
        "hf://datasets/openai/openai_humaneval/openai_humaneval/test-00000-of-00001.parquet"
    )

    # Get the first row
    first_row = dataset.iloc[0]
else:

    dataset = pd.read_json("/content/MBPP.jsonl", lines=True)
    first_row = dataset.iloc[0]
    print(first_row)
    print(first_row["test_list"])
    print(len(dataset))


#loading_model_with_api

In [ ]:
import getpass
import os
from openai import OpenAI
import os

os.environ["OPENAI_API_KEY"] = opeiai_token
os.environ["HF_TOKEN"] = hugging_face_token1


#hugging face

In [ ]:
import os
from openai import OpenAI

if FLAG_Model == "hugging_face":

    client = OpenAI(
        base_url="https://router.huggingface.co/v1",
        api_key=os.environ["HF_TOKEN"],
    )

    # completion = client.chat.completions.create(
    #     model="meta-llama/Llama-3.1-8B-Instruct:nscale",
    #     messages=[
    #         {
    #             "role": "user",
    #             "content": "What is the capital of France?"
    #         }
    #     ],
    # )

    # print(completion.choices[0].message)

    MODEL = "meta-llama/Llama-3.1-8B-Instruct:nscale"


#open AI

In [ ]:
from openai import OpenAI
import os

if FLAG_Model == "opeiai":

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))  # ← never hard-code this!

    MODEL = "gpt-3.5-turbo-1106"

    # MODEL = "gpt-4-turbo"
    # MODEL = "gpt-4-1106-vision-preview"  # or "gpt-4-1106-preview" or  gpt-4-1106-vision-preview

    # response = client.chat.completions.create(
    #     model=MODEL,          # or "gpt-4o", "o1-mini", "gpt-4.1", etc.
    #     messages=[
    #         {"role": "system", "content": "You are a helpful coding assistant."},
    #         {"role": "user",   "content": "Write a Python function that reverses a string without using [::-1]"}
    #     ],
    #     temperature=0.0,

    # )

    # # Most common way people print the answer
    # print(response.choices[0].message.content)

    # If you want to see the full response object:
    # print(response)


In [ ]:
import time


def generator(text: str):
    start_time = time.time()
    response = client.chat.completions.create(
        model=MODEL,  # or "gpt-4o", "o1-mini", "gpt-4.1", etc.
        messages=[
            {"role": "system", "content": "user"},
            {"role": "user", "content": f"{text}"},
        ],
        temperature=0.0,
        seed=42,
    )
    end_time = time.time()
    duration = end_time - start_time

    return (
        response.choices[0].message.content,
        response.usage.total_tokens,
        duration,
        response.system_fingerprint,
    )


In [ ]:
import os
import time

# from huggingface_hub import InferenceClient


def build_code_prompt(state):
    return f'generate a python code for this task: {state["prompt"]}'


def generator_coder(prompt: str):
    start_time = time.time()

    response = client.chat.completions.create(
        model=MODEL,  # or "gpt-4o", "o1-mini", "gpt-4.1", etc.
        messages=[{"role": "user", "content": f"{prompt}"}],
        temperature=0.0,
        seed=42,
    )

    end_time = time.time()
    duration = end_time - start_time
    return (
        response.choices[0].message.content,
        response.usage.total_tokens,
        duration,
        response.system_fingerprint,
    )


#get embedings

In [ ]:
from openai import OpenAI

client_embedding = OpenAI(api_key=opeiai_token)


def get_code_embedding(code: str) -> torch.Tensor:
    response = client_embedding.embeddings.create(
        model="text-embedding-3-large", input=f"{code}"
    )

    emb = response.data[0].embedding
    return torch.tensor(emb, dtype=torch.float32)
    # print(len(embedding))


code1 = """
function factorial(n) {
    if (n === 0 || n === 1) return 1;
    return n * factorial(n - 1);
}
"""
code = """function"""
r = get_code_embedding(code)


x = """'Counterfactual 1\n    Repair Strategy: Change the threshold comparison to absolute difference\n    Counterfactual Prompt:\n    ```python\n    def has_close_elements(numbers: List[float], threshold: float) -> bool:\n        for i in range(len(numbers)):\n            for j in range(i+1, len(numbers)):\n                if abs(numbers[i] - numbers[j]) < threshold:\n                    return True\n        return False\n    ```'"""
f = '''from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """'''

emb1 = get_code_embedding(x)
emb2 = get_code_embedding(f)


#cosine similarity

In [ ]:
def cosine_similarity(a: torch.Tensor, b: torch.Tensor) -> float:
    return torch.nn.functional.cosine_similarity(a, b, dim=0).item()


similarity = cosine_similarity(emb1, emb2)

print(f"Similarity: {similarity:.4f}")  # usually 0.82–0.93 range


#modules

##agent2

###AST

In [ ]:
import re


def extract_python_code(markdown_text: str):
    """
    Extracts and returns the first Python code block fenced with ```python
    If no such block exists, returns an empty string.
    """
    pattern = r"```python\s*(.*?)```"
    match = re.search(pattern, markdown_text, re.DOTALL | re.IGNORECASE)

    if match:
        return match.group(1).strip()  # Return the code inside the fences
    else:
        return markdown_text


# Example usage:
text_with_code = """
Here's some explanation.

```python
def hello():
    print("Hello, World!")
    ```
    kjhkjh
    """

extracted_code = extract_python_code(text_with_code)
print(extracted_code)


In [ ]:
import ast


def extract_function_signature(source_code):
    try:
        source_code = extract_python_code(source_code)
        tree = ast.parse(source_code)
        functions = []

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                # Reconstruct the signature: def name(params):
                args = node.args

                # Handle positional, keyword, *args, **kwargs
                param_parts = []

                # 1. Regular args
                for arg in args.posonlyargs:
                    param_parts.append(arg.arg)
                for arg in args.args:
                    param_parts.append(arg.arg)

                # 2. *args
                if args.vararg:
                    param_parts.append("*" + args.vararg.arg)

                # 3. Keyword-only args
                for arg in args.kwonlyargs:
                    param_parts.append(arg.arg)

                # 4. **kwargs
                if args.kwarg:
                    param_parts.append("**" + args.kwarg.arg)

                params_str = ", ".join(param_parts)
                signature = f"def {node.name}({params_str}):"

                functions.append({"name": node.name, "signature": signature})
                name = node.name
                signature = signature

        return name, signature
    except SyntaxError as e:
        return "False"


# === YOUR CODE ===
code1 = """
def find_similar_elements(tuple_list1, tuple_list2):
  similar_elements = []
  for tuple1 in tuple_list1:
    for tuple2 in tuple_list2:
      if set(tuple1) == set(tuple2):
        similar_elements.append(tuple1)
        break
  return similar_elements
"""
code2 = """def find_closest_numbers(numbers, threshold):
  if len(numbers) < 2:
    return False
  closest_pair = numbers[0:2]
  min_diff = abs(closest_pair[0] - closest_pair[1])
  for i in range(len(numbers) - 1):
    for j in range(i + 1, len(numbers)):
      diff = abs(numbers[i] - numbers[j])
      if diff < min_diff:
        min_diff = diff
        closest_pair = [numbers[i], numbers[j]]
  return closest_pair"""

code = '''import math


def poly(xs: list, x: float):
    """
    Evaluates polynomial with coefficients xs at point x.
    return xs[0] + xs[1] * x + xs[1] * x^2 + .... xs[n] * x^n
    """
    return sum([coeff * math.pow(x, i) for i, coeff in enumerate(xs)])


def find_zero(xs: list):
    """ xs are coefficients of a polynomial.
    find_zero find x such that poly(x) = 0.
    find_zero returns only only zero point, even if there are many.
    Moreover, find_zero only takes list xs having even number of coefficients
    and largest non zero coefficient as it guarantees
    a solution.
    >>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x
    -0.5
    >>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3
    1.0
    """
'''
# === EXTRACT ===
funcs = extract_function_signature(code)

# for f in funcs:
#     print("Name:", f['name'])
#     print("Signature:", f['signature'])
#     print()
print(funcs)


###build check function

In [ ]:
if FLAG == "mbpp":
    # only for mbpp dataset
    tests = first_row["test_list"]

    print(tests)

    def build_check_function(assert_lines, func_name="candidate"):
        """
        Convert a list like
            ['assert similar_elements(... ) == ...', ...]
        into
            def check(candidate):
                assert candidate(...) == ...
                ...

        Parameters
        ----------
        assert_lines : list[str]
            The original assert statements.
        func_name : str, optional
            Name of the function inside the generated check (default: "candidate").

        Returns
        -------
        str
            Complete Python code for the check-function.
        """
        # 1. Replace the old name with `func_name`
        converted = []
        for line in assert_lines:
            # split on the first '('
            head, tail = line.split("(", 1)
            # keep only "assert " and rebuild the call
            new_line = f"    assert {func_name}({tail}"
            converted.append(new_line)

        # 2. Build the whole function
        body = "\n".join(converted)
        code = f"""def check({func_name}):
{body}
  """
        return code

    print(build_check_function(tests))


###tester

In [ ]:
from typing import Any, Dict


class CodeTester:
    """A class to dynamically load and test a function from code strings."""

    def __init__(self, generated_code: str, check_code: str, entry_point: str):
        """Initialize the CodeTester with generated code, check code, and entry point."""
        self.generated_code = generated_code
        self.check_code = check_code
        self.entry_point = entry_point

    def load_function_from_code(self) -> Any:
        """Load a function from the stored generated code."""
        ns: Dict[str, Any] = {"__name__": "__generated__"}
        exec(self.generated_code, ns, ns)
        func = ns.get(self.entry_point)
        if not callable(func):
            raise RuntimeError(
                f"Entry point '{self.entry_point}' not found or not callable."
            )
        return func

    def load_check_from_string(self) -> Any:
        """Load a check function from the stored check code."""
        ns: Dict[str, Any] = {"__name__": "__check__"}
        exec(self.check_code, ns, ns)
        check_fn = ns.get("check")
        if not callable(check_fn):
            raise RuntimeError(
                "Function 'check' not found or not callable in check string."
            )
        return check_fn

    def run_tests(self) -> None:
        """Run the generated function against the check function and handle test results."""
        try:
            candidate = self.load_function_from_code()
            check_fn = self.load_check_from_string()
            check_fn(candidate)
            flag = "True"
            # print("All tests passed.")
        except AssertionError as e:
            flag = e
            # print("A test failed:", e)
        except Exception as e:
            flag = e
            # print("Execution error:", e)
        return flag

    def run_and_time(self) -> Dict[str, Any]:
        """Run the generated function, time its execution, and return results along with the execution time."""
        start_time = time.time()
        results = self.run_tests()
        end_time = time.time()
        execution_time = end_time - start_time
        return execution_time


# ---------------- Example usage ----------------

generated1_code = """
from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False
"""
# generated1_code=complete_code

generated_code = """
salam
from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False

"""
first_row = dataset.iloc[1]
check1_code = """def check(candidate):
    assert candidate((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)
    assert candidate((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)
    assert candidate((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)"""

check_code = """
def check(candidate):
    assert candidate(        [],       0.5) == False
"""
check_code1 = """def check(candidate):
    assert candidate([   [-1, 3, -3],   [2, -2, 4],   [-1, 4, 2] ]) == 1   """

generated_code1 = """
def minCostPath(cost):
# Write a function to find the minimum cost path to reach (m, n) from (0, 0) for the given cost matrix cost[][] and a position (m, n) in cost[][].
# The cost matrix cost[][] may contain negative values.

#```

#**
    m = len(cost)
    n = len(cost[0])
    dp = [[0 for _ in range(n)] for _ in range(m)]
    dp[0][0] = cost[0][0]
    for i in range(1, m):
        dp[i][0] = dp[i-1][0] + cost[i][0]
    for j in range(1, n):
        dp[0][j] = dp[0][j-1] + cost[0][j]
    for i in range(1, m):
        for j in range(1, n):
            dp[i][j] = min(dp[i-1][j], dp[i][j-1]) + cost[i][j]
    return dp[m-1][n-1]"""

# Create tester instance and run tests
tester = CodeTester(generated_code, check_code, entry_point="has_close_elements")
tester.run_tests()


In [ ]:
def extract_assert_lines(text):
    # Split the text into lines
    lines = text.splitlines()

    # Select lines containing 'assert'
    assert_lines = [line.strip() for line in lines if "assert" in line]

    # Build the new strings
    formatted_lines = [f"def check(candidate):\n    {line}" for line in assert_lines]

    return formatted_lines


def CodeTesterr(generated_code, check_code, entry_point):
    passed = 0
    check_code = extract_assert_lines(check_code)
    for i, check_codee in enumerate(check_code, 1):
        # Create tester instance and run tests
        tester = CodeTester(generated_code, check_codee, entry_point)
        tester.run_tests()
        if tester.run_tests() == "True":
            passed += 1

    return passed


generated_code1 = """
def minCostPath(cost):
# Write a function to find the minimum cost path to reach (m, n) from (0, 0) for the given cost matrix cost[][] and a position (m, n) in cost[][].
# The cost matrix cost[][] may contain negative values.

#```

#**
    m = len(cost)
    n = len(cost[0])
    dp = [[0 for _ in range(n)] for _ in range(m)]
    dp[0][0] = cost[0][0]
    for i in range(1, m):
        dp[i][0] = dp[i-1][0] + cost[i][0]
    for j in range(1, n):
        dp[0][j] = dp[0][j-1] + cost[0][j]
    for i in range(1, m):
        for j in range(1, n):
            dp[i][j] = min(dp[i-1][j], dp[i][j-1]) + cost[i][j]
    return dp[m-1][n-1]"""
check_code1 = """def check(candidate):
    assert candidate([   [-1, 3, -3],   [2, -2, 4],   [-1, 4, 2] ]) == 1   """
r = CodeTesterr(generated_code1, check_code1, "minCostPath")
print(r)


##agent3

###chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

text = """def closest_integer(value):
    ''' Create a function that takes a value (string) representing a number and returns the closest integer to it. If the number is equidistant from two integers, round it away from zero.
    Examples
    >>> closest_integer("10")
    10
    >>> closest_integer("15.3")
    15
    Note: Rounding away from zero means that if the given number is equidistant from two integers, the one you should return is the one that is the farthest from zero. For example closest_integer("14.5") should return 15 and closest_integer("-14.5") should return -15.
    '''
    pass  # your implementation here
"""
text1 = "Write a function to find the longest chain which can be formed from the given set of pairs."


def get_semantic_chunks(text: str) -> list[str]:

    splitter = RecursiveCharacterTextSplitter(
        # language=Language.PYTHON,
        chunk_size=200,  # ← tune this: 400–800 range often ideal
        chunk_overlap=0,  # keeps some context across boundaries
        separators=["\n\n", "\n", ".", ","],
    )

    # # Better alternative for bigger codebases
    # from llama_index.core.node_parser import CodeSplitter

    # splitter = CodeSplitter(
    #     language="python",
    #     chunk_lines=50,          # ~ lines per chunk
    #     chunk_lines_overlap=10,
    # )

    chunks = splitter.split_text(text)
    return chunks


chunks = get_semantic_chunks(text)
print(len(chunks))


In [ ]:
def split_into_three_parts(text):
    """
    Splits a string into 3 parts of as equal length as possible.
    Returns a list with 3 strings.
    """
    if not text or text.strip() == "":
        return ["", "", ""]

    words = text.split()
    n = len(words)

    part_size = n // 3
    remainder = n % 3

    # Calculate sizes for each chunk
    size1 = part_size + (1 if remainder > 0 else 0)
    size2 = part_size + (1 if remainder > 1 else 0)
    size3 = part_size  # remainder == 2 or 1 handled above

    part1 = words[:size1]
    part2 = words[size1 : size1 + size2]
    part3 = words[size1 + size2 :]

    return [" ".join(part1), " ".join(part2), " ".join(part3)]


# Example usage:
text = "This is a sample text that we want to split into three roughly equal parts."
parts = split_into_three_parts(text)
print(parts)


In [ ]:
def process_dict(items):
    n = len(items)
    if n < 3:
        return False
    if n == 3:
        return [str(x) for x in items]

    base = n // 3
    extra = n % 3

    result = []
    start = 0

    for i in range(3):
        size = base + (1 if i < extra else 0)
        chunk = [str(x) for x in items[start : start + size]]
        result.append("\n".join(chunk))
        start += size

    return result


di = ["hello", "hi", "boo", "app"]
r = process_dict(di)


##ablating

In [ ]:
def ablations_features(items):
    """
    Generate all possible feature combinations by removing exactly one item each time.
    Returns list of space-joined strings.

    Examples:
      ["red", "big", "car"]   →  ["big car", "red car", "red big"]
      ["A", "B"]              →  ["B", "A"]
      ["X"]                   →  [""]
      []                      →  []
      ["a","b","c","d"]      →  ["b c d", "a c d", "a b d", "a b c"]
    """
    if not items:
        return []

    result = []
    n = len(items)

    for i in range(n):
        # take all elements except the i-th one
        ablated = items[:i] + items[i + 1 :]
        # join with space (empty list → empty string)
        result.append(" ".join(ablated))

    return result


# ────────────────────────────────────────────────
# Usage examples
# ────────────────────────────────────────────────

features = ["speed", "power", "control"]

ablations = ablations_features(features)


In [ ]:
def to_percentages(values):
    total = sum(values)
    if total == 0:
        return [0 for _ in values]
    return [round((v / total) * 100, 2) for v in values]


# Example usage
nums = [0.00997382402420044, 0.017997324466705322, 0.017143964767456055]
percents = to_percentages(nums)
print(percents)  # Output: [16.66..., 33.33..., 50.0]


In [ ]:
def compare_numbers(numbers):
    """
    Compare numbers and return labels ordered as high, medium, low.
    Use the word equally when values are tied.
    Works with any number of values.
    """
    if len(numbers) < 2:
        return "لیست باید حداقل 2 عدد داشته باشد"

    # Check whether all numbers are equal
    if len(set(numbers)) == 1:
        return ["equally high"] * len(numbers)

    result = []

    for i, num in enumerate(numbers):
        # Count greater, smaller, and equal values
        greater_count = sum(
            1 for j, other in enumerate(numbers) if j != i and other > num
        )
        less_count = sum(1 for j, other in enumerate(numbers) if j != i and other < num)
        equal_count = sum(
            1 for j, other in enumerate(numbers) if j != i and other == num
        )

        # Determine the label
        if equal_count > 0:
            # At least one equal value exists
            if greater_count == 0:
                result.append("equally high")
            elif less_count == 0:
                result.append("equally low")
            else:
                result.append("equally medium")
        else:
            # No equal values exist
            if greater_count == 0:
                result.append("high")
            elif less_count == 0:
                result.append("low")
            else:
                result.append("medium")

    return result


# Previous tests (3 numbers)
print(compare_numbers([5, 3, 1]))  # ['high', 'medium', 'low']
print(compare_numbers([1, 3, 5]))  # ['low', 'medium', 'high']
print(compare_numbers([5, 5, 6]))  # ['equally high', 'equally high', 'low']
print(compare_numbers([5, 3, 5]))  # ['equally high', 'medium', 'equally high']
print(compare_numbers([3, 5, 5]))  # ['low', 'equally high', 'equally high']
print(compare_numbers([5, 5, 5]))  # ['equally', 'equally', 'equally']
print(compare_numbers([2, 2, 1]))  # ['equally high', 'equally high', 'low']

print("\n--- تست‌های جدید (بیشتر از 3 عدد) ---")
print(compare_numbers([5, 3, 1, 4]))  # ['high', 'medium', 'low', 'medium']
print(compare_numbers([1, 2, 3, 4, 5]))  # ['low', 'medium', 'medium', 'medium', 'high']
print(
    compare_numbers([5, 5, 3, 1])
)  # ['equally high', 'equally high', 'medium', 'low']
print(
    compare_numbers([10, 5, 5, 5, 1])
)  # ['high', 'equally medium', 'equally medium', 'equally medium', 'low']
print(compare_numbers([3, 3, 3, 3]))  # ['equally', 'equally', 'equally', 'equally']
print(
    compare_numbers([7, 2, 9, 2, 5])
)  # ['medium', 'equally low', 'high', 'equally low', 'medium']


#counterfacutal

In [ ]:
def levenshtein_distance(s1: str, s2: str) -> int:
    """
    Compute the Levenshtein (edit) distance between two strings.
    Returns the minimum number of single-character edits needed.

    Example:
        levenshtein_distance("kitten", "sitting")  → 3
    """
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)  # symmetry

    if len(s2) == 0:
        return len(s1)

    # Previous row
    previous_row = list(range(len(s2) + 1))

    for i, c1 in enumerate(s1, 1):
        # Current row starts with distance to empty string
        current_row = [i]

        for j, c2 in enumerate(s2, 1):
            # Cost = 0 if characters match, 1 otherwise
            cost = 0 if c1 == c2 else 1

            current_row.append(
                min(
                    current_row[j - 1] + 1,  # deletion
                    previous_row[j] + 1,  # insertion
                    previous_row[j - 1] + cost,  # substitution / match
                )
            )

        previous_row = current_row

    return previous_row[-1]


# ────────────────────────────────────────────────
# Quick test / demo
if __name__ == "__main__":
    test_cases = [
        # 3
        ("sunday", "saturday"),  # 3
        ("", "abc"),  # 3
        ("python", "python"),  # 0
        ("flaw", "lawn"),  # 2
        ("intention", "execution"),  # 5
        ("", ""),  # 0
    ]
    # (, )
    a = "kitten"
    b = "sitting"
    # for a, b in test_cases:
    dist = levenshtein_distance(a, b)
    # print(f"{a:12} → {b:12} : {dist}")
    print(dist)


In [ ]:
input_examples1 = [
    'import math\ndef poly(xs: list, x: float):\n    """\n    Evaluates polynomial with coefficients xs at point x.\n    return xs[0] + xs[1] * x + xs[1] * x^2 + .... xs[n] * x^n\n    """',
    'return sum([coeff * math.pow(x, i) for i, coeff in enumerate(xs)])\ndef find_zero(xs: list):\n    """ xs are coefficients of a polynomial.\n    find_zero find x such that poly(x) = 0.\n    find_zero returns only only zero point, even if there are many.',
    'Moreover, find_zero only takes list xs having even number of coefficients\n    and largest non zero coefficient as it guarantees\n    a solution.\n    >>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x\n-0.5\n    >>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3\n    1.0\n    """',
]

ablation_values = [0.00997382402420044, 0.017997324466705322, 0.017143964767456055]
fa_prompt = f"""You are an expert prompt engineer specializing in code generation refinement.

## Context
A code generation attempt has FAILED. We have performed feature ablation analysis
on the original prompt and identified which semantic chunks are most responsible
for the failure.

## Original Prompt (3 chunks)
Chunk 1: {input_examples1[0]}
Chunk 2: {input_examples1[1]}
Chunk 3: {input_examples1[2]}

## Importance Scores

- similarity; higher = more impact on failure)
- Chunk 1: {ablation_values[0]}
- Chunk 2: {ablation_values[1]}
- Chunk 3: {ablation_values[2]}

## Your Task
Generate EXACTLY 5 counterfactual prompts that minimally edit the original prompt
to fix the identified failure. Each counterfactual should:

1. **Target high-importance chunks** based on the scores above
2. **Apply different repair strategies**
3. **Be diverse from each other** (avoid near-duplicates)

## Return exactly 5 prompts using this format:
<new>
Counterfactual 1
Target Chunk 1 (Critical)
Repair Strategy:brief strategy description
Counterfactual Prompt:
```python
the complete counterfactual prompt text
```
<new>

REMEMBER you MUST ENCLOSE each counterfactual prompt within <new> tags.
"""

counterfactual_prompt = fa_prompt
# print(f"Counterfactual Prompt: {counterfactual_prompt}")
feature_attribution_explanation = fa_prompt

# Counterfactual processing starts here

# counterfactual_prompt , token_consumption_counter_prompt , duration_counter_prompt  = generator(counterfactual_prompt)


In [ ]:
counterfactual_prompt = '''
<new>
Counterfactual 1
Target Chunk 2 (Critical)
Repair Strategy: Use a more accurate method for evaluating the polynomial.
Counterfactual Prompt:
```python
import math
def poly(xs: list, x: float):
    """
    Evaluates polynomial with coefficients xs at point x.
    return xs[0] + xs[1] * x + xs[2] * x^2 + .... xs[n] * x^n
    """
return sum([coeff * x ** i for i, coeff in enumerate(xs)])
def find_zero(xs: list):
    """ xs are coefficients of a polynomial.
    find_zero find x such that poly(x) = 0.
    find_zero returns only only zero point, even if there are many.
Moreover, find_zero only takes list xs having even number of coefficients
and largest non zero coefficient as it guarantees
a solution.
>>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x
-0.5
>>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3
1.0
```
</new>

<new>
Counterfactual 2
Target Chunk 3 (Critical)
Repair Strategy: Modify the condition for finding the zero point.
Counterfactual Prompt:
```python
import math
def poly(xs: list, x: float):
    """
    Evaluates polynomial with coefficients xs at point x.
    return xs[0] + xs[1] * x + xs[2] * x^2 + .... xs[n] * x^n
    """
return sum([coeff * math.pow(x, i) for i, coeff in enumerate(xs)])
def find_zero(xs: list):
    """ xs are coefficients of a polynomial.
    find_zero find x such that poly(x) = 0.
    find_zero returns only only zero point, even if there are many.
Moreover, find_zero only takes list xs having an odd number of coefficients
and largest non zero coefficient as it guarantees
a solution.
>>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x
-0.5
>>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3
1.0
```
</new>

<new>
Counterfactual 3
Target Chunk 2 (Critical)
Repair Strategy: Use a more efficient method for evaluating the polynomial.
Counterfactual Prompt:
```python
import math
def poly(xs: list, x: float):
    """
    Evaluates polynomial with coefficients xs at point x.
    return xs[0] + xs[1] * x + xs[2] * x^2 + .... xs[n] * x^n
    """
return sum([coeff * x ** i for i, coeff in enumerate(xs)])
def find_zero(xs: list):
    """ xs are coefficients of a polynomial.
    find_zero find x such that poly(x) = 0.
    find_zero returns only only zero point, even if there are many.
Moreover, find_zero only takes list xs having even number of coefficients
and the largest non-zero coefficient as it guarantees a solution.
>>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x
-0.5
>>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3
1.0
```
</new>

<new>
Counterfactual 4
Target Chunk 1 (Critical)
Repair Strategy: Correct the polynomial evaluation formula.
Counterfactual Prompt:
```python
import math
def poly(xs: list, x: float):
    """
    Evaluates polynomial with coefficients xs at point x.
    return xs[0] + xs[1] * x + xs[2] * x**2 + .... xs[n] * x**n
    """
return sum([coeff * math.pow(x, i) for i, coeff in enumerate(xs)])
def find_zero(xs: list):
    """ xs are coefficients of a polynomial.
    find_zero find x such that poly(x) = 0.
    find_zero returns only only zero point, even if there are many.
Moreover, find_zero only takes list xs having even number of coefficients
and largest non zero coefficient as it guarantees
a solution.
>>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x
-0.5
>>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3
1.0
```
</new>

<new>
Counterfactual 5
Target Chunk 3 (Critical)
Repair Strategy: Relax the condition for finding the zero point.
Counterfactual Prompt:
```python
import math
def poly(xs: list, x: float):
    """
    Evaluates polynomial with coefficients xs at point x.
    return xs[0] + xs[1] * x + xs[2] * x^2 + .... xs[n] * x^n
    """
return sum([coeff * math.pow(x, i) for i, coeff in enumerate(xs)])
def find_zero(xs: list):
    """ xs are coefficients of a polynomial.
    find_zero find x such that poly(x) = 0.
    find_zero returns all zero points, if there are many.
Moreover, find_zero only takes list xs having even number of coefficients
and largest non zero coefficient as it guarantees
a solution.
>>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x
-0.5
>>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3
1.0
```
</new>'''


def extract_new_split(text: str) -> list[str]:
    if "<new>" in text:

        parts = text.split("<new>")
        results = []

        for part in parts[1:]:  # skip everything before first <new>
            if "</new>" in part:
                content, _ = part.split("</new>", 1)
                results.append(content.strip())
            # else: unclosed tag → we ignore or you can decide to take till end

        if results:
            return results
        else:
            pattern = r"(Counterfactual\s+\d+.*?)(?=Counterfactual\s+\d+|$)"

            blocks = re.findall(pattern, text, re.DOTALL | re.IGNORECASE)
            if blocks:

                return blocks
            else:
                return text


r = extract_new_split(counterfactual_prompt)


In [ ]:
def normalize_zero_one(values):
    min_val = min(values)
    max_val = max(values)
    if max_val == min_val:
        return [0 for _ in values]  # Avoid division by zero
    return [(v - min_val) / (max_val - min_val) for v in values]


ablation_values = [15, 2.1, 3.6]
r = normalize_zero_one(ablation_values)


#added

In [ ]:
def counterfact_without_xai(prompt, test, entry_point):

    fa_prompt = f"""You are an expert prompt engineer specializing in code generation refinement.

    ## Context
    A code generation attempt has FAILED.

    ## Your Task
    Generate EXACTLY 5 counterfactual prompts that minimally edit the original prompt
    to generate a better code and fix the failure.

    1. **Apply different repair strategies**
    2. **Be diverse from each other** (avoid near-duplicates)

    ##original prompt
    {prompt}

    ## Return exactly 5 prompts using this format:
    <new>
    Counterfactual 1
    Repair Strategy:brief strategy description
    Counterfactual Prompt:
    ```python
    the complete counterfactual prompt text
    ```
    </new>

    REMEMBER you MUST ENCLOSE each counterfactual prompt within <new> tags.
    """
    finger_print = []
    (
        counterfactual_prompt,
        token_consumption_counter_prompt,
        duration_counter_prompt,
        finger_print_prompt,
    ) = generator(fa_prompt)
    # c_prompts=counterfactual_prompt
    counterfactual_prompts = extract_new_split(counterfactual_prompt)
    finger_print.append(finger_print_prompt)

    sim = []
    edit_dis = []
    ablated_codeee = []
    number_passed_test_case = []
    counter_time = []
    token_consumption_ablation_coderr = 0
    duration_ablation_coderr = 0
    emb1 = get_code_embedding(prompt)
    resultt = []
    counter = 0
    edit_distance = []
    scores = []
    edit_d = 0
    score = 0
    if counterfactual_prompts is not None:

        for i, enum in enumerate(counterfactual_prompts):

            prompt1 = build_code_prompt({"prompt": enum})  # this is for mbpp

            # counterfactual_prompt=counterfactual_prompts[i]

            (
                ablated_code,
                token_consumption_ablation_coder,
                duration_ablation_coder,
                finger_print_coder,
            ) = generator_coder(f"{prompt1}")
            counter += 1

            # score=calculate_codebleu(ablated_code,state["complete_code"])
            ablated_code = extract_python_code(ablated_code)
            ablated_codeee.append(ablated_code)
            token_consumption_ablation_coderr += token_consumption_ablation_coder
            duration_ablation_coderr += duration_ablation_coder
            finger_print.append(finger_print_coder)

            try:

                f = extract_function_signature(ablated_code)
                if f != "False" and f[0] and f[1]:
                    entry_point = f[0]

                    function_signature = f[1]
                else:
                    entry_point = "none"
                    function_signature = "none"
            except:
                entry_point = entry_point
                function_signature = "none"

            resulttt = CodeTester(ablated_code, test, entry_point=entry_point)
            # resultt.append(result)

            result = resulttt.run_tests()
            if result == "True":

                edit_d = levenshtein_distance(prompt, enum)
                edit_distance.append(edit_d)

                emb2 = get_code_embedding(enum)
                score = cosine_similarity(emb1, emb2)
                sim.append(score)
                c_prompts = "True"
                resultt.append(result)
                break
            else:
                result = "False"
                resultt.append(result)
                c_prompts = "False"

            # timee=resulttt.run_and_time()
            # counter_time.append(timee)

            # number_passed_tc=CodeTesterr(ablated_code, test , entry_point= entry_point)
            # number_passed_test_case.append(number_passed_tc)

        resultt = len(resultt)
        ambiguty = "prompt is not ambigues."
    else:
        score = 0
        edit_d = 0
        c_prompts = 0
        resultt = 0
        finger_print = 0
        ambiguty = "prompt is ambigues."

    return (
        counterfactual_prompts,
        score,
        edit_d,
        c_prompts,
        resultt,
        finger_print,
        ambiguty,
    )


prompt = """
def minCostPath(cost):
# Write a function to find the minimum cost path to reach (m, n) from (0, 0) for the given cost matrix cost[][] and a position (m, n) in cost[][].
# The cost matrix cost[][] may contain negative values.

#```

"""
test = """def check(candidate):
    assert candidate([   [-1, 3, -3],   [2, -2, 4],   [-1, 4, 2] ]) == 1   """
entry_point = "minCostPath"

# counterfact , sim , edit_distance , c_prompts , resultt , fp =counterfact_without_xai(prompt , test , entry_point)


In [ ]:
import tiktoken


def count_tokens(data):

    model = "text-embedding-3-large"
    enc = tiktoken.encoding_for_model(model)

    # If the input is a string
    if isinstance(data, str):
        tokens = enc.encode(data)
        return len(tokens)

    # If the input is a list
    elif isinstance(data, list):
        total_tokens = 0
        for item in data:
            if not isinstance(item, str):
                raise ValueError("All list items must be strings")
            total_tokens += len(enc.encode(item))
        return total_tokens

    else:
        raise ValueError("Input must be a string or a list of strings")


print(count_tokens("Hello world"))

print(count_tokens(["Hello world", "This is a test sentence"]))


#send to telegram

In [ ]:
import pandas as pd
import requests
import time
import datetime
import os
from typing import Optional


def send_dataframe_to_telegram(number):

    bot_token = Bot_Token
    chat_id = Chat_Id
    filename_base = "/content/outputs.csv"
    message_text = f"Latest data update attached!row number :{number}"
    interval_seconds = 60  # every 1 minute
    run_forever = True
    max_iterations = None  # set to None for infinite

    if not os.path.exists(os.path.dirname(filename_base)) and os.path.dirname(
        filename_base
    ):
        os.makedirs(os.path.dirname(filename_base), exist_ok=True)

    # print(f"🚀 Telegram Auto-Sender started (every {interval_seconds} seconds)...")

    count = 0

    # while True:
    count += 1
    current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    try:
        # Save DataFrame to CSV
        filename = filename_base
        # df.to_csv(filename, index=False)

        # 1. Send Document
        url_document = f"https://api.telegram.org/bot{bot_token}/sendDocument"
        with open(filename, "rb") as file:
            files = {"document": file}
            data_doc = {
                "chat_id": chat_id,
                "caption": f"📤 Auto Update #{count}\n🕒 {current_time}",
            }
            requests.post(url_document, data=data_doc, files=files)

        # 2. Send Text Message
        url_message = f"https://api.telegram.org/bot{bot_token}/sendMessage"
        data_msg = {"chat_id": chat_id, "text": message_text, "parse_mode": "HTML"}
        requests.post(url_message, data=data_msg)

        # print(f"✅ Sent #{count} at {current_time}")

    except Exception as e:
        pass


#agentic workflow

In [ ]:
df = dataset.iloc[906:907]


In [ ]:
import pandas as pd
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional


import os
import torch
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import Image, display
import re
import csv
import os
import time
from google.colab import files
import traceback
import sys


class AgentState(TypedDict):
    row: int
    prompt: str
    complete_code: Optional[str]
    generated_code: Optional[str]
    test: str
    error_message: Optional[str]
    explanation: Optional[str]
    cycle_count: int
    new_prompt: Optional[str]
    entry_point: str
    function_signature: Optional[str]
    test_function: Optional[str]
    token_consumption_coder: Optional[int]
    duration_coder: Optional[int]
    token_consumption_chunks: Optional[int]
    duration_chunks: Optional[int]
    token_consumption_counter_prompt: Optional[int]
    duration_counter_prompt: Optional[int]
    token_consumption_counetr_coder: Optional[int]
    duration_counter_coder: Optional[int]
    token_consumption_debug: Optional[int]
    duration_debug: Optional[int]
    duration_kernelshap: Optional[int]
    token_consumption_ablation_coder: Optional[int]
    duration_ablation_coder: Optional[int]
    sim: Optional[float]
    edit_dis: Optional[int]
    path: str
    harmful_chunk: Optional[str]
    duration_workflow: Optional[int]
    duration_workflow1: Optional[int]
    start_time: Optional[int]
    end_time: Optional[int]
    start_time1: Optional[int]
    end_time1: Optional[int]
    compiler: Optional[str]
    api_call: int
    counterfactual_correction: Optional[str]
    ablated_codeee: Optional[str]
    token_consumption_ablation_coderr: Optional[int]
    duration_ablation_coderr: Optional[int]
    semantic_chunk: Optional[str]
    number_counter: Optional[int]
    ablation_counter: Optional[int]
    edit_dis_avg: Optional[int]
    sim_avg: Optional[int]
    all_tokens: Optional[int]
    embedding_tokens: Optional[int]
    first_prompt: Optional[str]
    finger_print_coder: Optional[str]
    finger_print_ablation_coder: Optional[str]

    ablated_codee: Optional[str]
    input_examples: Optional[str]
    ablated_features: Optional[str]
    ablation_values: Optional[str]
    feature_attribution_explanation: Optional[str]
    counterfactual_prompts: Optional[str]
    scores: Optional[str]
    resultt_simple: Optional[str]
    sim_simple: Optional[str]
    edit_distance_simple: Optional[str]
    c_prompts: Optional[str]
    sim_simple_avg: Optional[str]
    edit_distance_simple_avg: Optional[str]
    counterfact_simple: Optional[str]
    resultt_simple: Optional[str]
    finger_print_counter_prompt: Optional[str]
    fp_code_counter: Optional[str]
    counterfactual_error: Optional[str]


def coder_agent(state: AgentState) -> AgentState:
    state["api_call"] = 0
    state["token_consumption_coder"] = 0
    state["token_consumption_ablation_coder"] = 0
    state["token_consumption_counter_prompt"] = 0
    state["token_consumption_ablation_coderr"] = 0
    state["all_tokens"] = 0
    state["number_counter"] = 0
    state["ablation_counter"] = 0
    state["embedding_tokens"] = 0
    state["start_time"] = time.time()

    prompt = build_code_prompt(state)

    complete_code, token_consumption_coder, duration_coder, finger_print_coder = (
        generator_coder(f"{prompt}")
    )
    state["api_call"] += 1

    complete_code = extract_python_code(complete_code)

    state["token_consumption_coder"] = token_consumption_coder
    state["duration_coder"] = duration_coder
    state["finger_print_coder"] = finger_print_coder

    state["complete_code"] = complete_code

    f = extract_function_signature(state["complete_code"])
    if f != "False" and f[0] and f[1]:
        state["entry_point"] = f[0]

        state["function_signature"] = f[1]
    else:
        state["entry_point"] = "none"
        state["function_signature"] = "none"

    return state


def tester_agent(state: AgentState) -> AgentState:

    if FLAG == "mbpp":
        test_cases = build_check_function(state["test"])
    else:
        test_cases = state["test"]

    state["test_function"] = test_cases

    result = CodeTester(
        state["complete_code"], state["test_function"], entry_point=state["entry_point"]
    )

    result = result.run_tests()

    if result == "True":

        state["error_message"] = result
        state["end_time"] = time.time()
        state["duration_workflow"] = state["end_time"] - state["start_time"]

        state["all_tokens"] = state["token_consumption_coder"]
        new_row = [
            state["row"],
            state["cycle_count"],
            state["prompt"],
            state["complete_code"],
            state["test"],
            state["error_message"],
            state["token_consumption_coder"],
            state["duration_coder"],
            state["duration_workflow"],
            state["api_call"],
            state["all_tokens"],
            state["finger_print_coder"],
        ]

        with open(state["path"], "a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(new_row)
        return state
    else:

        state["error_message"] = "False"
        state["compiler"] = result
        if state["error_message"] == "False" and state["cycle_count"] == 0:
            state["end_time"] = time.time()
            state["duration_workflow"] = state["end_time"] - state["start_time"]

            state["all_tokens"] = state["token_consumption_coder"]

            tokens = 0
            api_calls = 0

            new_row = [
                state["row"],
                state["cycle_count"],
                state["prompt"],
                state["complete_code"],
                state["test"],
                state["error_message"],
                state["token_consumption_coder"],
                state["duration_coder"],
                state["duration_workflow"],
                api_calls,
                tokens,
                state["finger_print_coder"],
                state["compiler"],
            ]

            with open(state["path"], "a", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(new_row)
        state["cycle_count"] += 1

        return state


def route_after_agent(state: AgentState) -> AgentState:
    if state["error_message"] == "True":

        return "Pass"
    else:

        if state["cycle_count"] >= 4:

            return "Pass"

        else:

            return "Fail"


def xai_explanier_agent(state: AgentState) -> AgentState:
    state["start_time1"] = time.time()

    prompt = build_code_prompt(state)

    list_of_chunks = get_semantic_chunks(state["prompt"])
    r = process_dict(list_of_chunks)

    if r is False:
        list_of_chunks = split_into_three_parts(state["prompt"])
        state["semantic_chunk"] = "False"
    else:
        list_of_chunks = r
        state["semantic_chunk"] = "True"

    input_examples = list_of_chunks

    input_examples1 = input_examples

    token_consumption_ablation_coder1 = 0
    duration_ablation_coder1 = 0

    ablated_features = ablations_features(input_examples1)

    ablation_values = []
    ablated_codee = []
    fp_ablation_coder = []

    embeddings_lower2 = get_code_embedding(state["complete_code"])
    state["embedding_tokens"] = state["embedding_tokens"] + count_tokens(
        state["complete_code"]
    )
    state["api_call"] += 1

    for i, item in enumerate(ablated_features, 1):
        prompt = build_code_prompt({"prompt": item})

        (
            ablated_code,
            token_consumption_ablation_coder,
            duration_ablation_coder,
            finger_print_ablation_coder,
        ) = generator_coder(f"{prompt}")
        state["api_call"] += 1
        state["ablation_counter"] += 1

        ablated_code = extract_python_code(ablated_code)
        ablated_codee.append(ablated_code)
        fp_ablation_coder.append(finger_print_ablation_coder)

        try:

            f = extract_function_signature(ablated_code)
            if f != "False" and f[0] and f[1]:
                entry_point = f[0]

                function_signature = f[1]
            else:
                entry_point = "none"
                function_signature = "none"
        except:
            entry_point = state["entry_point"]
            function_signature = "none"

        result = CodeTester(
            ablated_code, state["test_function"], entry_point=entry_point
        )

        result = result.run_tests()

        if result != "chert":

            state["harmful_chunk"] = "False"

            embeddings_lower1 = get_code_embedding(ablated_code)
            state["embedding_tokens"] = state["embedding_tokens"] + count_tokens(
                ablated_code
            )
            state["api_call"] += 1

            sim = cosine_similarity(embeddings_lower1, embeddings_lower2)
            sim = 1 - sim
            ablation_values.append(sim)
            token_consumption_ablation_coder1 += token_consumption_ablation_coder
            duration_ablation_coder1 += duration_ablation_coder
            state["token_consumption_ablation_coder"] = (
                token_consumption_ablation_coder1
            )
            state["duration_ablation_coder"] = duration_ablation_coder1
            state["finger_print_ablation_coder"] = fp_ablation_coder
            ablation_qualities = compare_numbers(ablation_values)

    fa_prompt = f"""You are an expert prompt engineer specializing in code generation refinement.

## Context
A code generation attempt has FAILED. split the prompt into 3 chunks. we calculate which chunk is more important in generating the failed code.

## Original Prompt (3 chunks)
Chunk 1: {input_examples1[0]}
Chunk 2: {input_examples1[1]}
Chunk 3: {input_examples1[2]}

## Importance of chunks
- Chunk 1: {ablation_qualities[0]} Impact
- Chunk 2: {ablation_qualities[1]} Impact
- Chunk 3: {ablation_qualities[2]} Impact

## Your Task
Generate EXACTLY 5 counterfactual prompts that edit the original prompt minimal as possible
to generate a better code and fix the failure. Each counterfactual should:

1. **Target high-impact chunks** based on the impacts above
2. **Apply different repair strategies**
3. **Be diverse from each other** (avoid near-duplicates)

## Return exactly 5 prompts using this format:
<new>
Counterfactual 1
Target Chunk 1 (Critical)
Repair Strategy:brief strategy description
Counterfactual Prompt:
```python
the complete counterfactual prompt text
```
</new>

REMEMBER you MUST ENCLOSE each counterfactual prompt within <new> tags.
"""
    if state["cycle_count"] == 1 and FLAG == "humaneval":
        (
            counterfact_simple,
            sim_simple,
            edit_distance_simple,
            c_prompts,
            resultt_simple,
            finger_print_simple_counter,
            ambiguty,
        ) = counterfact_without_xai(
            state["prompt"], state["test_function"], state["entry_point"]
        )
        sim_simple_avg = sim_simple
        edit_distance_simple_avg = edit_distance_simple
        emb1 = get_code_embedding(state["first_prompt"])

    else:
        counterfact_simple = 0
        resultt_simple = 0
        sim_simple = 0
        edit_distance_simple = 0
        c_prompts = 0
        sim_simple_avg = 0
        edit_distance_simple_avg = 0
        finger_print_simple_counter = 0
        ambiguty = 0

    counterfactual_prompt = fa_prompt

    feature_attribution_explanation = fa_prompt

    (
        counterfactual_prompt,
        token_consumption_counter_prompt,
        duration_counter_prompt,
        finger_print_counter_prompt,
    ) = generator(counterfactual_prompt)
    state["api_call"] += 1
    state["token_consumption_counter_prompt"] = token_consumption_counter_prompt
    state["duration_counter_prompt"] = duration_counter_prompt

    counterfactual_prompts = extract_new_split(counterfactual_prompt)

    sim = 0
    edit_dis = []
    ablated_codeee = []
    number_passed_test_case = []
    counter_time = []
    token_consumption_ablation_coderr = 0
    duration_ablation_coderr = 0
    edit_distance = 0
    scores = []
    fp_code_counter = []

    if counterfactual_prompts is not None:

        for i, enum in enumerate(counterfactual_prompts, 0):

            prompt = build_code_prompt({"prompt": enum})

            (
                ablated_code,
                token_consumption_ablation_coder,
                duration_ablation_coder,
                finger_print_code_counter,
            ) = generator_coder(f"{prompt}")
            state["api_call"] += 1
            state["number_counter"] += 1

            ablated_code = extract_python_code(ablated_code)
            ablated_codeee.append(ablated_code)
            token_consumption_ablation_coderr += token_consumption_ablation_coder
            duration_ablation_coderr += duration_ablation_coder
            fp_code_counter.append(finger_print_code_counter)

            try:

                f = extract_function_signature(ablated_code)
                if f != "False" and f[0] and f[1]:
                    entry_point = f[0]

                    function_signature = f[1]
                else:
                    entry_point = "none"
                    function_signature = "none"
            except:
                entry_point = state["entry_point"]
                function_signature = "none"

            resultt = CodeTester(
                ablated_code, state["test_function"], entry_point=entry_point
            )

            result = resultt.run_tests()
            timee = resultt.run_and_time()
            counter_time.append(timee)

            number_passed_tc = CodeTesterr(
                ablated_code, state["test_function"], entry_point=entry_point
            )
            number_passed_test_case.append(number_passed_tc)

            if result == "True":

                state["error_message"] = "True"

                state["counterfactual_correction"] = "True"
                state["complete_code"] = ablated_code
                old_prompt = state["prompt"]
                state["prompt"] = enum
                if state["cycle_count"] == 1:
                    edit_distance = levenshtein_distance(state["first_prompt"], enum)
                else:
                    edit_distance = 0

                if state["cycle_count"] == 1:

                    emb2 = get_code_embedding(enum)
                    sim = cosine_similarity(emb1, emb2)

                break

            else:
                old_prompt = state["prompt"]
                state["counterfactual_correction"] = "False"

        edit_distanceee = []
        for i in range(len(counterfactual_prompts)):

            edit_d = levenshtein_distance(old_prompt, counterfactual_prompts[i])
            edit_distanceee.append(edit_d)

        if state["counterfactual_correction"] != "True":
            number_passed_test_casee = normalize_zero_one(number_passed_test_case)
            counter_timee = normalize_zero_one(counter_time)
            edit_dd = normalize_zero_one(edit_distanceee)
            for i in range(len(number_passed_test_casee)):

                score = (
                    number_passed_test_casee[i] - counter_timee[i] + edit_dd[i]
                ) / 3
                scores.append(score)
            best_index = scores.index(max(scores))

            strings = counterfactual_prompts
            best_string = strings[best_index]
            state["prompt"] = best_string

        state["sim"] = sim

        state["sim_avg"] = 0
        state["edit_dis"] = edit_distance

        state["edit_dis_avg"] = 0
        state["token_consumption_ablation_coderr"] = token_consumption_ablation_coderr
        state["duration_ablation_coderr"] = duration_ablation_coderr
        state["ablated_codeee"] = ablated_codeee

    else:
        state["sim"] = 0

        state["sim_avg"] = 0
        state["edit_dis"] = 0

        state["edit_dis_avg"] = 0
        state["token_consumption_ablation_coderr"] = 0
        state["duration_ablation_coderr"] = 0
        state["ablated_codeee"] = 0
        state["counterfactual_correction"] = 0
        state["prompt"] = counterfactual_prompt
        state["counterfactual_error"] = "True"
        state["number_counter"] = 0
        counterfactual_prompt = counterfactual_prompts

    state[
        "explanation"
    ] = f"""Here's what we learned from analyzing your original prompt:

  We split your prompt into 3 meaningful parts and tested what happens when we remove each one:
  Chunk 1: {input_examples1[0]}
  Chunk 2: {input_examples1[1]}
  Chunk 3: {input_examples1[2]}

  Importance scores (how much removing that chunk changes the code that gets generated):
  Chunk 1: {ablation_values[0]}
  Chunk 2: {ablation_values[1]}
  Chunk 3: {ablation_values[2]}

  This means:
  The part with the highest value is the one most responsible for the code going wrong.
  Removing it caused the biggest change in behavior (usually for the worse).

  Based on this the error message we saw , we created 5 slightly improved version of your prompt.to flip the result and pass the test case.
  New (counterfactual) prompts we recommend trying:
  {counterfactual_prompt}

  the edit distance between the original prompt and counterfacual prompt is {state['edit_dis']}

  The new prompt focuses most fixes on the most important chunk(s) usually just 1 or 2 small changes like clearer wording, added constraints, or an extra example.
  the final prompt is :
  {state['prompt']}

  and this is the final code:
  {state['complete_code']}

  """
    state["end_time1"] = time.time()
    state["duration_workflow1"] = state["end_time1"] - state["start_time1"]

    state["all_tokens"] = (
        state["all_tokens"]
        + state["token_consumption_counter_prompt"]
        + state["token_consumption_ablation_coderr"]
        + state["embedding_tokens"]
    )

    new_row = [
        state["row"],
        state["cycle_count"],
        state["prompt"],
        state["complete_code"],
        state["test"],
        state["error_message"],
        state["token_consumption_coder"],
        state["duration_coder"],
        state["duration_workflow"],
        state["api_call"],
        state["all_tokens"],
        state["finger_print_coder"],
        state["compiler"],
        state["finger_print_ablation_coder"],
        ablated_codee,
        state["duration_workflow1"],
        input_examples,
        ablated_features,
        state["harmful_chunk"],
        state["token_consumption_ablation_coder"],
        state["duration_ablation_coder"],
        state["explanation"],
        ablation_values,
        state["semantic_chunk"],
        state["ablation_counter"],
        state["embedding_tokens"],
        feature_attribution_explanation,
        counterfactual_prompts,
        state["token_consumption_counter_prompt"],
        state["duration_counter_prompt"],
        state["sim"],
        state["edit_dis"],
        state["ablated_codeee"],
        state["token_consumption_ablation_coderr"],
        state["duration_ablation_coderr"],
        state["counterfactual_correction"],
        state["number_counter"],
        scores,
        resultt_simple,
        sim_simple,
        edit_distance_simple,
        state["edit_dis_avg"],
        state["sim_avg"],
        c_prompts,
        sim_simple_avg,
        edit_distance_simple_avg,
        counterfact_simple,
        resultt_simple,
        finger_print_counter_prompt,
        fp_code_counter,
        finger_print_simple_counter,
        ambiguty,
        state["counterfactual_error"],
    ]

    with open(state["path"], "a", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(new_row)
    return state


def route_after_agentt(state: AgentState) -> AgentState:
    if (
        state["counterfactual_correction"] == "True"
        or state["harmful_chunk"] == "True"
        or state["counterfactual_error"] == "True"
    ):
        return "Pass"
    else:
        return "Fail"


def create_workflow():

    workflow = StateGraph(AgentState)

    workflow.add_node("coder", coder_agent)
    workflow.add_node("tester", tester_agent)
    workflow.add_node("xai_explanier", xai_explanier_agent)

    workflow.set_entry_point("coder")
    workflow.add_edge("coder", "tester")
    workflow.add_conditional_edges(
        "tester", route_after_agent, {"Fail": "xai_explanier", "Pass": END}
    )
    workflow.add_conditional_edges(
        "xai_explanier", route_after_agentt, {"Fail": "coder", "Pass": END}
    )

    compiled_workflow = workflow.compile()

    try:

        display(Image(compiled_workflow.get_graph().draw_mermaid_png()))
    except Exception:
        print(
            "Mermaid diagram rendering failed. Ensure required dependencies (e.g., pyppeteer) are installed."
        )

    return compiled_workflow


def process_dataset(df):

    workflow = create_workflow()
    output_file = "/content/results.csv"

    import time

    last_time = time.time()

    for index, row in df.iterrows():
        if FLAG == "mbpp":

            TEST = row["test_list"]
            PROMPT = row["text"]
        else:
            PROMPT = row["prompt"]
            TEST = row["test"]

        initial_state = {
            "row": index + 1,
            "prompt": PROMPT,
            "first_prompt": PROMPT,
            "code": None,
            "generated_code": None,
            "test": TEST,
            "entry_point": None,
            "error_message": None,
            "explanation": None,
            "cycle_count": 0,
            "new_prompt": None,
            "function_signature": None,
            "complete_code": None,
            "test_function": None,
            "token_consumption_coder": None,
            "duration_coder": None,
            "token_consumption_chunks": None,
            "duration_chunks": None,
            "token_consumption_counter_prompt": None,
            "duration_counter_prompt": None,
            "token_consumption_counetr_coder": None,
            "duration_counter_coder": None,
            "token_consumption_debug": None,
            "duration_debug": None,
            "duration_kernelshap": None,
            "duration_counterfactual": None,
            "token_consumption_ablation_coder": None,
            "duration_ablation_coder": None,
            "sim": None,
            "edit_dis": None,
            "harmful_chunk": None,
            "duration_workflow": None,
            "duration_workflow1": None,
            "start_time": None,
            "end_time": None,
            "start_time1": None,
            "end_time1": None,
            "compiler": None,
            "api_call": 0,
            "ablated_codee": None,
            "duration_ablation_coderr": None,
            "token_consumption_ablation_coderr": None,
            "counterfactual_correction": None,
            "number_counter": 0,
            "semantic_chunk": None,
            "ablation_counter": 0,
            "sim_avg": 0,
            "edit_dis_avg": 0,
            "all_tokens": 0,
            "embedding_tokens": 0,
            "finger_print_coder": None,
            "counterfactual_error": None,
            "finger_print_ablation_coder": None,
            "path": "/content/outputs.csv",
        }

        try:

            final_state = workflow.invoke(initial_state)
            print(f"\nFinal State for Row {index + 1}:")

            if FLAG_TELE == "True" and time.time() - last_time > 300:

                send_dataframe_to_telegram(index + 1)
                last_time = time.time()

        except Exception as e:

            error_log = traceback.format_exc()

            new_row = [index + 1, type(e).__name__, str(e), error_log]

            with open("/content/error.csv", "a", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(new_row)

            print(f"⚠️ Row {index + 1} failed: {type(e).__name__}: {e}")
            continue


process_dataset(df)


In [ ]:
from google.colab import files

files.download("/content/outputs.csv")
send_dataframe_to_telegram(10000000)
